In [2]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
import shutil
import yaml
import torch
import csv
import random
from pathlib import Path
from ultralytics import YOLO

## Import the best model for the training

In [ ]:
# Setup paths and load model
PROJECT_ROOT = Path(os.getcwd()).parent
results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Project root: {PROJECT_ROOT}")
print(f"Results directory: {results_dir}")

# Load best model - check multiple possible locations
possible_paths = [
    PROJECT_ROOT / 'runs/yolov8n_vehicle_detection2/weights/best.pt',
    PROJECT_ROOT / '../runs/yolov8n_vehicle_detection2/weights/best.pt',
]

model = None
for model_path in possible_paths:
    if model_path.exists():
        model = YOLO(str(model_path))
        print(f"✅ Trained model loaded from: {model_path}")
        break

# Fallback to base yolov8n model if trained model not found
if model is None:
    base_model_path = PROJECT_ROOT / 'yolov8n.pt'
    if base_model_path.exists():
        model = YOLO(str(base_model_path))
        print(f"⚠️  Using base model (not trained): {base_model_path}")
        print("Note: For better results, train the model using 'antoine train and test the model.ipynb'")
    else:
        print("❌ No model found. Please train the model first.")


Project root: c:\Users\alera\Desktop\ENTPE\ICVIS\project\ICV_project
Results directory: c:\Users\alera\Desktop\ENTPE\ICVIS\project\ICV_project\results
✅ Trained model loaded from: c:\Users\alera\Desktop\ENTPE\ICVIS\project\ICV_project\runs\detect\weights\best.pt


## Compute flow and density metrics

In [4]:
import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict, deque
from ultralytics import YOLO
from tqdm import tqdm
import math

# --- Configurazione Percorsi (Basata su "antoine flow and density.ipynb") ---
# Si assume che il notebook sia nella cartella 'notebooks'
PROJECT_ROOT = Path(os.getcwd()).parent 
DATA_PROCESSED = PROJECT_ROOT / "data_processed"
TEST_IMAGES_DIR = DATA_PROCESSED / "test" / "images"
RESULTS_DIR = PROJECT_ROOT / "results" / "offense_detection"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📂 Project Root: {PROJECT_ROOT}")
print(f"📂 Input Images: {TEST_IMAGES_DIR}")
print(f"📂 Output Video: {RESULTS_DIR}")

# Controllo device
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"⚡ Running on: {device}")

📂 Project Root: c:\Users\alera\Desktop\ENTPE\ICVIS\project\ICV_project
📂 Input Images: c:\Users\alera\Desktop\ENTPE\ICVIS\project\ICV_project\data_processed\test\images
📂 Output Video: c:\Users\alera\Desktop\ENTPE\ICVIS\project\ICV_project\results\offense_detection
⚡ Running on: cuda


In [5]:
# --- Caricamento Modello ---
# Cerca il modello best.pt nei percorsi standard di YOLO
possible_paths = [
    PROJECT_ROOT / 'runs/detect/weights/best.pt',
    PROJECT_ROOT / 'runs/detect/weights/best.pt', # Percorso visto nel tuo log
    PROJECT_ROOT / 'runs/detect/train/weights/best.pt'
]

model = None
for model_path in possible_paths:
    if model_path.exists():
        model = YOLO(str(model_path))
        print(f"✅ Modello addestrato caricato da: {model_path}")
        break

if model is None:
    print("⚠️ Modello addestrato non trovato. Scarico yolov8n.pt base (meno accurato).")
    model = YOLO("yolov8n.pt")

✅ Modello addestrato caricato da: c:\Users\alera\Desktop\ENTPE\ICVIS\project\ICV_project\runs\detect\weights\best.pt


In [6]:
class TrafficOffenseMonitor:
    def __init__(self, fps=25, pixels_per_meter=15, speed_limit_kmh=50):
        """
        Args:
            fps: Frame per secondo del video/sequenza
            pixels_per_meter: Fattore di calibrazione (quanti pixel sono 1 metro). 
                              DA TARARE in base alla sequenza specifica.
            speed_limit_kmh: Limite velocità per segnalare infrazione.
        """
        self.fps = fps
        self.ppm = pixels_per_meter
        self.speed_limit = speed_limit_kmh
        
        # Storico posizioni: {track_id: deque([(frame, x, y), ...])}
        self.positions = defaultdict(lambda: deque(maxlen=fps*2)) # Tiene 2 secondi di storia
        
        # Stato sorpassi: {(id_A, id_B): 'behind'/'ahead'}
        self.relative_positions = {}
        
        # Output per visualizzazione
        self.current_speeds = {} # {id: speed_kmh}
        self.active_offenses = {'speeding': set(), 'right_pass': set()}

    def update(self, tracks, frame_idx):
        """
        Processa i risultati del tracking per il frame corrente.
        tracks: Risultato di model.track()
        """
        # Pulisci infrazioni momentanee
        self.active_offenses['right_pass'] = set()
        
        if tracks.boxes is None or not tracks.boxes.id:
            return

        boxes = tracks.boxes.xyxy.cpu().numpy()
        ids = tracks.boxes.id.cpu().numpy().astype(int)
        
        current_frame_pos = {} # id: (cx, cy)

        for box, track_id in zip(boxes, ids):
            x1, y1, x2, y2 = box
            cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
            current_frame_pos[track_id] = (cx, cy)
            
            # Aggiungi allo storico
            self.positions[track_id].append((frame_idx, cx, cy))
            
            # --- 1. RILEVAMENTO VELOCITÀ (SPEEDING) ---
            speed = self._calculate_speed(track_id)
            self.current_speeds[track_id] = speed
            
            if speed > self.speed_limit:
                self.active_offenses['speeding'].add(track_id)
            elif track_id in self.active_offenses['speeding']:
                # Rimuovi se scende sotto il limite (con un po' di isteresi opzionale)
                self.active_offenses['speeding'].discard(track_id)

        # --- 2. RILEVAMENTO SORPASSO A DESTRA (PASS ON RIGHT) ---
        # Logica: Se A era DIETRO B e ora è AVANTI a B, e A è a DESTRA di B.
        ids_list = list(current_frame_pos.keys())
        for i in range(len(ids_list)):
            for j in range(i + 1, len(ids_list)):
                id_a, id_b = ids_list[i], ids_list[j]
                self._check_overtake(id_a, id_b, current_frame_pos)

    def _calculate_speed(self, track_id):
        history = self.positions[track_id]
        if len(history) < 5: return 0.0
        
        # Calcola delta su 0.5 secondi (o meno se inizio traccia)
        frames_back = min(len(history) - 1, int(self.fps / 2))
        curr = history[-1]
        prev = history[-len(history) + frames_back] # Prendi il punto più vecchio nel buffer utile
        
        # Distanza Euclidea in pixel
        pixel_dist = math.sqrt((curr[1] - prev[1])**2 + (curr[2] - prev[2])**2)
        
        # Conversione
        meters = pixel_dist / self.ppm
        seconds = (curr[0] - prev[0]) / self.fps
        
        if seconds == 0: return 0.0
        return (meters / seconds) * 3.6 # m/s a km/h

    def _check_overtake(self, id_a, id_b, curr_pos):
        ay, by = curr_pos[id_a][1], curr_pos[id_b][1]
        
        # Definiamo chi è avanti (assumiamo Y cresce verso il basso)
        # Se le auto vanno verso l'alto (Y diminuisce), invertire la logica
        # Qui assumiamo standard camera view (auto si allontanano -> Y diminuisce, vanno in alto)
        # Quindi Y minore = Avanti.
        a_ahead = ay < by
        
        state = 'ahead' if a_ahead else 'behind'
        pair = tuple(sorted((id_a, id_b)))
        
        if pair in self.relative_positions:
            prev_state = self.relative_positions[pair]
            
            if prev_state != state: # SCAMBIO DI POSIZIONE (Sorpasso)
                # Chi ha sorpassato chi?
                passer = id_a if state == 'ahead' else id_b
                passed = id_b if passer == id_a else id_a
                
                # Verifica posizione laterale (X)
                # Assumiamo X cresce verso destra.
                px = curr_pos[passer][0] # Passer X
                vx = curr_pos[passed][0] # Victim X
                
                # Se chi passa ha X maggiore, sta passando a destra
                if px > vx:
                    # Registra l'evento per questo frame
                    self.active_offenses['right_pass'].add(passer)
                    
        self.relative_positions[pair] = state

In [7]:
import cv2
import math
from tqdm import tqdm
from collections import defaultdict, deque

# ==========================================
# 1. CONFIGURAZIONE
# ==========================================
# Inserisci qui il nome esatto della cartella della sequenza che vuoi analizzare
NOME_SEQUENZA = "MVI_39031"  
# Esempi comuni: "MVI_40905", "MVI_20011", "MVI_63563", etc.

# ==========================================
# 2. CLASSE PER LE INFRAZIONI
# ==========================================
class TrafficOffenseMonitor:
    def __init__(self, fps=25, pixels_per_meter=15, speed_limit_kmh=50):
        self.fps = fps
        self.ppm = pixels_per_meter
        self.speed_limit = speed_limit_kmh
        self.positions = defaultdict(lambda: deque(maxlen=fps*2)) 
        self.relative_positions = {}
        self.current_speeds = {} 
        self.active_offenses = {'speeding': set(), 'right_pass': set()}

    def update(self, tracks, frame_idx):
        self.active_offenses['right_pass'] = set() # Reset per frame
        
        if tracks.boxes is None or tracks.boxes.id is None:
            return

        boxes = tracks.boxes.xyxy.cpu().numpy()
        ids = tracks.boxes.id.cpu().numpy().astype(int)
        current_frame_pos = {} 

        for box, track_id in zip(boxes, ids):
            x1, y1, x2, y2 = box
            cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
            current_frame_pos[track_id] = (cx, cy)
            self.positions[track_id].append((frame_idx, cx, cy))
            
            # 1. Calcolo Velocità
            speed = self._calculate_speed(track_id)
            self.current_speeds[track_id] = speed
            
            if speed > self.speed_limit:
                self.active_offenses['speeding'].add(track_id)
            elif track_id in self.active_offenses['speeding']:
                self.active_offenses['speeding'].discard(track_id)

        # 2. Controllo Sorpasso a Destra
        ids_list = list(current_frame_pos.keys())
        for i in range(len(ids_list)):
            for j in range(i + 1, len(ids_list)):
                self._check_overtake(ids_list[i], ids_list[j], current_frame_pos)

    def _calculate_speed(self, track_id):
        history = self.positions[track_id]
        if len(history) < 5: return 0.0
        frames_back = min(len(history) - 1, int(self.fps / 2))
        curr = history[-1]
        prev = history[-len(history) + frames_back]
        pixel_dist = math.sqrt((curr[1] - prev[1])**2 + (curr[2] - prev[2])**2)
        meters = pixel_dist / self.ppm
        seconds = (curr[0] - prev[0]) / self.fps
        return (meters / seconds) * 3.6 if seconds > 0 else 0.0

    def _check_overtake(self, id_a, id_b, curr_pos):
        # Assumiamo che Y diminuisca andando avanti (veicoli si allontanano)
        a_ahead = curr_pos[id_a][1] < curr_pos[id_b][1] 
        state = 'ahead' if a_ahead else 'behind'
        pair = tuple(sorted((id_a, id_b)))
        
        if pair in self.relative_positions:
            if self.relative_positions[pair] != state: # Cambio posizione
                passer = id_a if state == 'ahead' else id_b
                passed = id_b if passer == id_a else id_a
                # Se chi passa ha X maggiore (è a destra), è infrazione
                if curr_pos[passer][0] > curr_pos[passed][0]:
                    self.active_offenses['right_pass'].add(passer)
        self.relative_positions[pair] = state

# ==========================================
# 3. FUNZIONE DI ELABORAZIONE VIDEO
# ==========================================
def analyze_single_sequence(sequence_name):
    seq_path = TEST_IMAGES_DIR / sequence_name
    
    # Controllo esistenza
    if not seq_path.exists():
        print(f"❌ Errore: La sequenza '{sequence_name}' non esiste in {TEST_IMAGES_DIR}")
        print("Cartelle disponibili:", [d.name for d in TEST_IMAGES_DIR.iterdir() if d.is_dir()][:5], "...")
        return

    images = sorted(list(seq_path.glob("*.jpg")) + list(seq_path.glob("*.png")))
    if not images:
        print("❌ Nessuna immagine trovata nella cartella.")
        return

    print(f"🎬 Inizio analisi sequenza: {sequence_name}")
    print(f"   Numero frame: {len(images)}")

    # Inizializza Monitor
    # pixels_per_meter: 8.0 è un valore empirico per DETRAC (auto lontane)
    monitor = TrafficOffenseMonitor(fps=25, pixels_per_meter=8.0, speed_limit_kmh=60)

    # Setup Video Output
    first_img = cv2.imread(str(images[0]))
    h, w = first_img.shape[:2]
    out_path = RESULTS_DIR / f"analysis_{sequence_name}.mp4"
    writer = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*'mp4v'), 25, (w, h))

    # Ciclo sui frame
    for i, img_path in enumerate(tqdm(images, desc="Processing")):
        frame = cv2.imread(str(img_path))
        
        # Tracking YOLO
        results = model.track(frame, persist=True, verbose=False, tracker="bytetrack.yaml")[0]
        
        # Aggiorna logica infrazioni
        monitor.update(results, i)
        
        # Disegno visuale
        if results.boxes.id is not None:
            boxes = results.boxes.xyxy.cpu().numpy()
            ids = results.boxes.id.cpu().numpy().astype(int)
            
            for box, track_id in zip(boxes, ids):
                x1, y1, x2, y2 = map(int, box)
                
                # Default: Verde
                color = (0, 255, 0) 
                info_text = f"ID:{track_id} {int(monitor.current_speeds.get(track_id, 0))}km/h"
                
                # Rosso se Speeding
                if track_id in monitor.active_offenses['speeding']:
                    color = (0, 0, 255)
                    info_text += " SPEED"
                
                # Arancione se Sorpasso a Destra
                if track_id in monitor.active_offenses['right_pass']:
                    color = (0, 165, 255)
                    info_text += " RX-PASS"
                
                # Disegna
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.rectangle(frame, (x1, y1 - 20), (x1 + 200, y1), color, -1)
                cv2.putText(frame, info_text, (x1, y1 - 5), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

        writer.write(frame)

    writer.release()
    print(f"\n✅ Analisi completata!")
    print(f"📂 Video salvato in: {out_path}")

# ==========================================
# 4. ESECUZIONE
# ==========================================
analyze_single_sequence(NOME_SEQUENZA)

🎬 Inizio analisi sequenza: MVI_39031
   Numero frame: 1470


Processing:   0%|          | 1/1470 [00:01<34:54,  1.43s/it]

: 